# Raman Spectroscopy Data Analysis (Master Notebook)

**Process once, reuse forever** — Automatically creates and reuses dark-subtracted Excel output.

## 🔧 CELL 1 — Import Libraries (never change)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 📂 CELL 2 — 👉 CHANGE ONLY THIS PATH (EVERY TIME)

In [2]:
# 🔴 CHANGE THIS TO ANY ONE DATA FOLDER
data_folder = Path(
    r"c:\Users\sukes\Downloads\sukesh NAM Raman data\python-notebook\5sec-power5-80 and 2set"
)

## 📄 CELL 3 — Output Excel Name (fixed)

In [3]:
output_excel = data_folder / "combined_dark_subtracted.xlsx"
dark_file_name = "dark.csv"   # set to None if you don't have dark file

## 🧠 CELL 4 — Logic: Reuse Output if Exists

In [4]:
if output_excel.exists():
    print("📄 Output already exists. Loading Excel...")
    df = pd.read_excel(output_excel)

else:
    print("⚙️ Output not found. Processing CSV files...")

    csv_files = sorted([
        f for f in data_folder.glob("*.csv")
        if f.name != dark_file_name
    ])

    if len(csv_files) == 0:
        raise ValueError("❌ No CSV files found in folder")

    # Load dark spectrum
    if dark_file_name and (data_folder / dark_file_name).exists():
        dark_df = pd.read_csv(data_folder / dark_file_name, header=None)
        dark_intensity = dark_df.iloc[:, 1].values
    else:
        dark_intensity = None

    X = None
    Y_list = []

    for file in csv_files:
        df_temp = pd.read_csv(file, header=None)

        if X is None:
            X = df_temp.iloc[:, 0].values  # Raman shift

        intensity = df_temp.iloc[:, 1].values

        if dark_intensity is not None:
            intensity = intensity - dark_intensity

        Y_list.append(intensity)

    Y = np.array(Y_list).T

    columns = ["x1"] + [f"y{i+1}" for i in range(Y.shape[1])]
    data = np.column_stack([X, Y])

    df = pd.DataFrame(data, columns=columns)

    df.to_excel(output_excel, index=False)
    print("✅ Output created and saved:")
    print(output_excel)

⚙️ Output not found. Processing CSV files...


ParserError: Error tokenizing data. C error: Expected 2 fields in line 99, saw 15


## 👀 CELL 5 — View the Data (confirmation)

In [ ]:
df.head()

## 📈 CELL 6 — Quick Overlay Plot (to "see the data")

In [5]:
X = df["x1"].values

plt.figure(figsize=(8, 5))

for col in df.columns[1:]:
    plt.plot(X, df[col], alpha=0.4)

plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Intensity (dark-subtracted)")
plt.title(f"Overlay spectra: {data_folder.name}")
plt.show()

NameError: name 'df' is not defined

---

## 🔬 ADVANCED PROCESSING PIPELINE

The following cells perform professional-grade spectral processing.

## 🔹 CELL 7 — Extract X and Y Matrices

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X = df["x1"].values
Y = df.iloc[:, 1:].values   # y1, y2, ...
num_spectra = Y.shape[1]

print(f"Loaded {num_spectra} spectra")

## 🔹 CELL 8 — Baseline Correction (Asymmetric Least Squares)

In [ ]:
from scipy import sparse
from scipy.sparse.linalg import spsolve

def baseline_als(y, lam=1e5, p=0.01, niter=10):
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L-2))
    w = np.ones(L)

    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L)
        Z = W + lam * D @ D.T
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y < z)

    return z

# Apply baseline correction
Y_baseline = np.zeros_like(Y)

for i in range(num_spectra):
    base = baseline_als(Y[:, i])
    Y_baseline[:, i] = Y[:, i] - base

print("✅ Baseline correction complete")

## 🔹 CELL 9 — Savitzky–Golay Smoothing

In [ ]:
from scipy.signal import savgol_filter

Y_smooth = savgol_filter(
    Y_baseline,
    window_length=11,
    polyorder=3,
    axis=0
)

print("✅ Smoothing complete")

## 🔹 CELL 10 — Normalization

In [ ]:
# Option A: Max peak normalization (recommended for comparison)
Y_norm = Y_smooth / np.max(Y_smooth, axis=0)

# Option B: Area normalization (uncomment if preferred)
# area = np.trapz(Y_smooth, X, axis=0)
# Y_norm = Y_smooth / area

print("✅ Normalization complete")

## 🔹 CELL 11 — Overlay Processed Spectra (visual validation)

In [ ]:
plt.figure(figsize=(8, 5))

for i in range(num_spectra):
    plt.plot(X, Y_norm[:, i], alpha=0.4)

plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Normalized Intensity")
plt.title("Processed Raman Spectra Overlay")
plt.show()

## 🔹 CELL 12 — Peak Finder (NAM candidate peaks)

In [ ]:
from scipy.signal import find_peaks

all_peaks = []

for i in range(num_spectra):
    peaks, props = find_peaks(
        Y_norm[:, i],
        height=0.05,
        distance=20
    )
    all_peaks.append(peaks)

print(f"✅ Peak detection complete for {num_spectra} spectra")

## 🔹 CELL 13 — Identify Consistent (stable) NAM Peaks

In [ ]:
from collections import Counter

peak_positions = []

for peaks in all_peaks:
    peak_positions.extend(np.round(X[peaks]))

counter = Counter(peak_positions)

# Peaks present in ≥70% of spectra
stable_peaks = [
    p for p, c in counter.items()
    if c >= 0.7 * num_spectra
]

stable_peaks = sorted(stable_peaks)

print(f"✅ Found {len(stable_peaks)} stable peaks:")
print(stable_peaks)

## 🔹 CELL 14 — Peak Stability Plot (NAM validation)

In [ ]:
if len(stable_peaks) > 0:
    peak_cm = stable_peaks[0]   # choose main NAM peak
    idx = np.argmin(np.abs(X - peak_cm))

    peak_intensity = Y_norm[idx, :]

    plt.figure(figsize=(8, 5))
    plt.plot(range(1, num_spectra+1), peak_intensity, "o-")
    plt.xlabel("Spectrum Number")
    plt.ylabel("Normalized Peak Intensity")
    plt.title(f"Peak Stability at {peak_cm} cm⁻¹")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("⚠️ No stable peaks found. Adjust detection parameters in Cell 12.")

## 🔹 CELL 15 — Peak Intensity vs Integration Time

In [ ]:
import re

# Extract time from folder name
match = re.search(r'(\d+)sec', data_folder.name)
if match:
    integration_time = int(match.group(1))
else:
    integration_time = None

print("Integration time:", integration_time, "sec")

# Calculate mean peak intensity for this folder
if len(stable_peaks) > 0:
    peak_cm = stable_peaks[0]
    idx = np.argmin(np.abs(X - peak_cm))
    peak_intensity = Y_norm[idx, :]
    mean_peak_intensity = np.mean(peak_intensity)
    
    print(f"Mean peak intensity at {peak_cm} cm⁻¹: {mean_peak_intensity:.4f}")
    print(f"\n💡 Store this result: ({integration_time}, {mean_peak_intensity:.4f})")
else:
    print("⚠️ No stable peaks to analyze")

## 🔹 CELL 16 — Final Comparison Plot (manual aggregation)

Run this after collecting data from all folders. Update the values based on Cell 15 output.

In [ ]:
# Example data - update with your actual values from Cell 15
times = [5, 10, 15, 20, 25]
intensities = [0.32, 0.51, 0.71, 0.89, 1.00]  # Replace with actual values

plt.figure(figsize=(8, 5))
plt.plot(times, intensities, "o-", linewidth=2, markersize=8)
plt.xlabel("Integration Time (s)")
plt.ylabel("Normalized Peak Intensity")
plt.title("NAM Peak Intensity vs Integration Time")
plt.grid(alpha=0.3)
plt.show()